# WRDS Data Download
**项目：** LLM Signal × 财报公告 CAR  
**对应文档：** Design Doc / Data Doc  
**范围：** 1996–2026，使用月度更新版 `crsp_m_stock` 覆盖最新数据  
**输出：** 全部保存为 parquet → `./data/`


## 0. Setup

In [10]:
import wrds
import pandas as pd
import time
from pathlib import Path

OUT = Path("./data")
OUT.mkdir(exist_ok=True)

START = "1996-01-01"
END = pd.Timestamp.today().strftime("%Y-%m-%d")

conn = wrds.Connection()
print("WRDS 连接成功")


WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
WRDS 连接成功


In [4]:
END

'2026-07-24'

## 1. 跨库连接表

三张表本身不对应任何回归变量，但后续所有跨库 merge 都依赖它们。

| 表 | 作用 |
|---|---|
| `crsp.ccmxpf_lnkhist` | PERMNO ↔ GVKEY |
| `wrdsapps.ibcrsphist` | I/B/E/S ticker ↔ PERMNO |
| `crsp.stksecurityinfohist` | 股票类型筛选 + SIC + CUSIP↔PERMNO |


In [4]:
# 1.1 PERMNO ↔ GVKEY
df = conn.raw_sql("""
    SELECT gvkey, lpermno AS permno, lpermco AS permco,
           linktype, linkprim, linkdt, linkenddt
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LU','LC')
      AND linkprim IN ('P','C')
""")
df.to_parquet(OUT / "link_crsp_compustat.parquet", index=False)
print(f"link_crsp_compustat: {len(df):,} rows")


link_crsp_compustat: 33,324 rows


**`link_crsp_compustat`（PERMNO ↔ GVKEY 连接表）**

| 字段 | English Full Name | 中文解释 |
|---|---|---|
| gvkey | Global Company Key | Compustat 公司标识符 |
| permno | Permanent Number | CRSP 股票永久标识符 |
| permco | Permanent Company Number | CRSP 公司永久标识符 |
| linktype | Link Type | 连接类型（LU/LC = 主要可靠连接） |
| linkprim | Link Primary Flag | 是否主要连接标记（P=主要, C=次要但可靠） |
| linkdt | Link Start Date | 连接生效起始日期 |
| linkenddt | Link End Date | 连接生效结束日期 |


In [5]:
# 1.2 I/B/E/S ticker ↔ PERMNO
df = conn.raw_sql("""
    SELECT ticker, permno, sdate, edate
    FROM wrdsapps.ibcrsphist
    WHERE score <= 2
""")
df.to_parquet(OUT / "link_ibes_crsp.parquet", index=False)
print(f"link_ibes_crsp: {len(df):,} rows")


link_ibes_crsp: 30,080 rows


**`link_ibes_crsp`（I/B/E/S ↔ CRSP 连接表）**

| 字段 | English Full Name | 中文解释 |
|---|---|---|
| ticker | I/B/E/S Ticker Symbol | I/B/E/S 股票代码 |
| permno | Permanent Number | CRSP 股票永久标识符 |
| sdate | Start Date | 该连接映射有效起始日 |
| edate | End Date | 该连接映射有效结束日 |


In [11]:
# 1.3 股票信息历史（股票类型 / SIC / 交易所 / CUSIP）
df = conn.raw_sql("""
    SELECT permno, permco, cusip, siccd,
           primaryexch, conditionaltype, tradingstatusflg,
           secinfostartdt, secinfoenddt,
           sharetype, securitytype, securitysubtype,
           usincflg, issuertype
    FROM crsp_m_stock.stksecurityinfohist
""")
df.to_parquet(OUT / "crsp_security_info.parquet", index=False)
print(f"crsp_security_info: {len(df):,} rows")

crsp_security_info: 193,968 rows


**`crsp_security_info`（股票信息历史表）**

| 字段 | English Full Name | 中文解释 |
|---|---|---|
| permno | Permanent Number | CRSP 股票永久标识符 |
| permco | Permanent Company Number | CRSP 公司永久标识符 |
| cusip | CUSIP | 证券统一识别码 |
| siccd | Standard Industrial Classification Code | 标准行业分类代码（用于行业固定效应 ψ_j） |
| primaryexch | Primary Exchange | 主要上市交易所（N=NYSE, A=AMEX, Q=NASDAQ） |
| conditionaltype | Conditional Type | 股票条件类型标记 |
| tradingstatusflg | Trading Status Flag | 交易状态标记 |
| secinfostartdt | Security Info Start Date | 该条股票信息的生效起始日 |
| secinfoenddt | Security Info End Date | 该条股票信息的生效结束日 |


## 2. CRSP 日频股票数据

**构造目标：** CAR 的 $R_{i,t}$、SIZE（dlycap）、TURN/IO 分母（shrout）

按年分批下载，自动跳过已存在的文件。使用 `crsp_m_stock.dsf_v2`（月度更新版，覆盖到2026年）。


In [8]:
years = list(range(1996, 2027))

for yr in years:
    fpath = OUT / f"crsp_daily_{yr}.parquet"
    if fpath.exists():
        print(f"  已存在，跳过: crsp_daily_{yr}")
        continue
    try:
        df = conn.raw_sql(f"""
            SELECT permno, dlycaldt, dlyret, dlyprc,
                   dlycap, shrout, dlyvol, primaryexch
            FROM crsp_m_stock.dsf_v2
            WHERE dlycaldt BETWEEN '{yr}-01-01' AND '{yr}-12-31'
        """)
        df.to_parquet(fpath, index=False)
        print(f"  crsp_daily_{yr}: {len(df):,} rows | "
              f"{df['dlycaldt'].min()} ~ {df['dlycaldt'].max()}")
    except Exception as e:
        print(f"  ✗ crsp_daily_{yr}: {e}")
    time.sleep(1)

print("CRSP 日频下载完成")


  crsp_daily_1996: 2,247,238 rows | 1996-01-02 ~ 1996-12-31
  crsp_daily_1997: 2,322,956 rows | 1997-01-02 ~ 1997-12-31
  crsp_daily_1998: 2,290,093 rows | 1998-01-02 ~ 1998-12-31
  crsp_daily_1999: 2,178,713 rows | 1999-01-04 ~ 1999-12-31
  crsp_daily_2000: 2,135,112 rows | 2000-01-03 ~ 2000-12-29
  crsp_daily_2001: 1,966,502 rows | 2001-01-02 ~ 2001-12-31
  crsp_daily_2002: 1,864,523 rows | 2002-01-02 ~ 2002-12-31
  crsp_daily_2003: 1,766,860 rows | 2003-01-02 ~ 2003-12-31
  crsp_daily_2004: 1,738,143 rows | 2004-01-02 ~ 2004-12-31
  crsp_daily_2005: 1,739,983 rows | 2005-01-03 ~ 2005-12-30
  crsp_daily_2006: 1,737,027 rows | 2006-01-03 ~ 2006-12-29
  crsp_daily_2007: 1,773,957 rows | 2007-01-03 ~ 2007-12-31
  crsp_daily_2008: 1,779,140 rows | 2008-01-02 ~ 2008-12-31
  crsp_daily_2009: 1,696,873 rows | 2009-01-02 ~ 2009-12-31
  crsp_daily_2010: 1,687,284 rows | 2010-01-04 ~ 2010-12-31
  crsp_daily_2011: 1,706,623 rows | 2011-01-03 ~ 2011-12-30
  crsp_daily_2012: 1,701,545 rows | 2012

**`crsp_daily_{year}`（CRSP 日频股票数据，来自 `crsp_m_stock.dsf_v2`）**

| 字段 | English Full Name | 中文解释 | 对应 Design Doc 变量 |
|---|---|---|---|
| permno | Permanent Number | CRSP 股票永久标识符 | 公司标识 i |
| dlycaldt | Daily Calendar Date | 交易日期 | 日期 d/t |
| dlyret | Daily Return | 日收益率 | R_{i,t}（CAR 构造核心） |
| dlyprc | Daily Price | 收盘价 | — |
| dlycap | Daily Market Cap | 当日市值（已算好，无需自算 price×shares） | SIZE |
| shrout | Shares Outstanding | 流通股本 | TURN / IO 分母 |
| dlyvol | Daily Volume | 当日成交量 | TURN 分子 |
| primaryexch | Primary Exchange | 主要交易所 | Breakpoint universe 筛选 |


In [10]:
import pandas as pd
from pathlib import Path

OUT = Path("./data")

df = pd.read_parquet(OUT / "crsp_daily_2026.parquet")
df.head()

,permno,dlycaldt,dlyret,dlyprc,dlycap,shrout,dlyvol,primaryexch
0,10026,2026-01-02,0.00166,90.52,1719517.92,18996,194864.0,Q
1,10028,2026-01-02,-0.103886,11.99,311296.37,25963,121724.0,A
2,10032,2026-01-02,0.035578,152.23,4066519.99,26713,158257.0,Q
3,10044,2026-01-02,0.01579,1.93,17950.93,9301,27436.0,Q
4,10065,2026-01-02,-0.003002,23.25,2821085.25,121337,319680.0,N


In [11]:
18996*90.52

1719517.92

## 3. CRSP 月频股票数据

**构造目标：** SIZE（年初/6月底市值）、BM 分母（12月底市值）、TURN（月均成交量）


In [12]:
df = conn.raw_sql(f"""
    SELECT permno, mthcaldt, mthret, mthprc,
           mthcap, shrout, mthvol, primaryexch,
           mthcumfacpr, mthcumfacshr
    FROM crsp_m_stock.msf_v2
    WHERE mthcaldt BETWEEN '1994-01-01' AND '{END}'
""")
df.to_parquet(OUT / "crsp_monthly.parquet", index=False)
print(f"crsp_monthly: {len(df):,} rows | "
      f"{df['mthcaldt'].min()} ~ {df['mthcaldt'].max()}")


crsp_monthly: 3,091,767 rows | 1994-01-31 ~ 2026-06-30


**`crsp_monthly`（CRSP 月频股票数据，来自 `crsp_m_stock.msf_v2`）**

| 字段 | English Full Name | 中文解释 | 对应 Design Doc 变量 |
|---|---|---|---|
| permno | Permanent Number | CRSP 股票永久标识符 | 公司标识 i |
| mthcaldt | Monthly Calendar Date | 月末交易日期 | Formation 时点锚点 |
| mthret | Monthly Return | 月收益率 | — |
| mthprc | Monthly Price | 月末收盘价 | — |
| mthcap | Monthly Market Cap | 月末市值 | SIZE（年初/6月底）、BM 分母（12月底） |
| shrout | Shares Outstanding | 流通股本 | TURN 分母 |
| mthvol | Monthly Volume | 月成交量 | TURN 分子 |
| primaryexch | Primary Exchange | 主要交易所 | Breakpoint universe 筛选 |
| mthcumfacpr | Cumulative Factor to Adjust Price | 累积价格调整因子（拆股/股票股利） | **SUE 分母**：`mthprc / mthcumfacpr` 折到与 I/B/E/S EPS 相同的最新基准 |
| mthcumfacshr | Cumulative Factor to Adjust Shares | 累积股数调整因子 | **TURN**：跨拆股期间调整月成交量口径，`mthvol × mthcumfacshr` 后再除以当前 shrout |


**为什么需要这两个因子（实测依据）：**

I/B/E/S 的 actual / forecast 已被统一折算到**最新拆股基准**——AAPL 2012Q2 在 `ibes_actuals`
里是 `0.4393`，而当年披露的是 `$12.30`，相差 28 倍 = 2014 年 7:1 × 2020 年 4:1。
而 CRSP 的 `mthprc` 是**当年的历史价格**（AAPL 2012 年约 $600）。

若直接用 `(e − F) / mthprc`，拆过股的公司 SUE 会被系统性缩小，且每家缩小的倍数不同
（取决于此后拆股次数）——这会污染 SUE 的截面排序，进而污染十分位分组。
正确做法是把价格折到同一基准：`P_adj = mthprc / mthcumfacpr`。

注意 HLT 2009 的清洗规则"剔除拆股调整前股价 < $1"用的是**未调整**价格，
所以 `mthprc` 原值同样要保留 —— 两个都用得上。

In [8]:
df2 = pd.read_parquet(OUT / "crsp_monthly.parquet")
df2.head()

,permno,mthcaldt,mthret,mthprc,mthcap,shrout,mthvol,primaryexch
0,10001,1996-01-31,-0.026667,9.125,20814.13,2281,16845.0,Q
1,10002,1996-01-31,-0.035714,13.5,40527.0,3002,2775.0,Q
2,10009,1996-01-31,-0.009174,13.5,31576.5,2339,11623.0,Q
3,10011,1996-01-31,0.085106,12.75,99386.25,7795,837248.0,Q
4,10012,1996-01-31,-0.104762,5.875,92619.38,15765,2439779.0,Q


## 4. Compustat 年度基本面

**构造目标：** BE = SEQ + TXDITC − PS → BM（控制变量 + 25组 benchmark 分组）


In [13]:
df = conn.raw_sql(f"""
    SELECT gvkey, datadate, fyear, fyr,
           seq, ceq, pstk, at, lt,
           txditc, pstkrv, pstkl, sich
    FROM comp.funda
    WHERE indfmt = 'INDL'
      AND datafmt = 'STD'
      AND popsrc = 'D'
      AND consol = 'C'
      AND datadate BETWEEN '1993-01-01' AND '{END}'
""")
df.to_parquet(OUT / "compustat_annual.parquet", index=False)
print(f"compustat_annual: {len(df):,} rows | "
      f"{df['datadate'].min()} ~ {df['datadate'].max()}")


compustat_annual: 388,641 rows | 1993-01-31 ~ 2026-06-30


**`compustat_annual`（Compustat 年度基本面，来自 `comp.funda`）**

| 字段 | English Full Name | 中文解释 | 对应 Design Doc 变量 |
|---|---|---|---|
| gvkey | Global Company Key | Compustat 公司标识符 | 公司标识（需转 PERMNO） |
| datadate | Data Date | 财年结束日期 | Formation 时点 |
| fyear | Fiscal Year | 财政年度 | — |
| fyr | Fiscal Year End Month | 财年结束月份 | — |
| seq | Stockholders Equity | 股东权益 | BE 分量（首选） |
| ceq | Common/Ordinary Equity | 普通股权益 | BE 备用分量（seq 缺失时） |
| pstk | Preferred Stock — Total | 优先股账面价值（总） | BE 备用分量 / PS 备选 |
| at | Total Assets | 总资产 | BE 备用分量（seq、ceq+pstk 均缺失时） |
| lt | Total Liabilities | 总负债 | BE 备用分量 |
| txditc | Deferred Taxes and Investment Tax Credit | 递延所得税及投资税收抵免 | BE 分量 |
| pstkrv | Preferred Stock Redemption Value | 优先股赎回价值 | PS 首选取值 |
| pstkl | Preferred Stock Liquidating Value | 优先股清算价值 | PS 次选取值 |
| sich | Historical SIC Code | 历史 SIC 行业代码 | 行业固定效应 ψ_j 备选来源 |


In [14]:
df = pd.read_parquet(OUT / "compustat_annual.parquet")

# 每列缺失比例
print(df[['seq','ceq','pstk','txditc','pstkrv','pstkl']].isnull().mean())

# 总行数和总缺失数对照看
print(df[['seq','ceq','pstk','txditc','pstkrv','pstkl']].isnull().sum())
print(f"总行数: {len(df):,}")

seq       0.204947
ceq       0.206594
pstk      0.207037
txditc    0.288508
pstkrv    0.212397
pstkl     0.212613
dtype: float64
seq        79651
ceq        80291
pstk       80463
txditc    112126
pstkrv     82546
pstkl      82630
dtype: int64
总行数: 388,641


In [15]:
df.head(10)


,gvkey,datadate,fyear,fyr,seq,ceq,pstk,at,lt,txditc,pstkrv,pstkl,sich
0,001004,1993-05-31,1992,5,189.216,189.216,0.0,365.151,175.935,38.0,0.0,0.0,5080
1,001004,1994-05-31,1993,5,189.488,189.488,0.0,417.626,228.138,39.0,0.0,0.0,5080
2,001004,1995-05-31,1994,5,197.119,197.119,0.0,425.814,228.695,30.66,0.0,0.0,5080
3,001004,1996-05-31,1995,5,204.635,204.635,0.0,437.846,233.211,30.68,0.0,0.0,5080
4,001004,1997-05-31,1996,5,269.259,269.259,0.0,529.584,260.325,32.56,0.0,0.0,5080
5,001004,1998-05-31,1997,5,300.85,300.85,0.0,670.559,369.709,36.85,0.0,0.0,5080
6,001004,1999-05-31,1998,5,326.035,326.035,0.0,726.63,400.595,44.87,0.0,0.0,5080
7,001004,2000-05-31,1999,5,339.515,339.515,0.0,740.998,401.483,56.02,0.0,0.0,5080
8,001004,2001-05-31,2000,5,340.212,340.212,0.0,701.854,361.642,55.063,0.0,0.0,5080
9,001004,2002-05-31,2001,5,310.235,310.235,0.0,710.199,399.964,30.601,0.0,0.0,5080


In [16]:
seq_filled = df['seq'].notna()
ceq_pstk_filled = df['ceq'].notna() & df['pstk'].notna()
at_lt_filled = df['at'].notna() & df['lt'].notna()

can_compute_be = seq_filled | ceq_pstk_filled | at_lt_filled
print(f"能算出BE的比例: {can_compute_be.mean():.2%}")
print(f"完全算不出BE的行数: {(~can_compute_be).sum():,}")

能算出BE的比例: 79.51%
完全算不出BE的行数: 79,616


In [17]:
missing = df[~can_compute_be].copy()
missing['year'] = pd.to_datetime(missing['datadate']).dt.year

# 看缺失是不是集中在某些年份（早期数据覆盖率通常更差）
print(missing['year'].value_counts().sort_index())

# 缺失公司数占当年总公司数的比例
year_missing_rate = df.assign(
    year=pd.to_datetime(df['datadate']).dt.year,
    missing=~can_compute_be
).groupby('year')['missing'].mean()
print(year_missing_rate)

year
1993    1842
1994    1713
1995    1404
1996    1326
1997    1375
1998    1079
1999     928
2000     881
2001     912
2002     990
2003    1032
2004    1081
2005    1167
2006    1382
2007    1762
2008    1833
2009    1937
2010    2179
2011    2599
2012    2506
2013    2454
2014    2598
2015    2746
2016    2988
2017    3099
2018    3364
2019    3630
2020    3834
2021    4108
2022    4425
2023    4725
2024    5261
2025    6291
2026     165
Name: count, dtype: int64
year
1993    0.160930
1994    0.144569
1995    0.112672
1996    0.104615
1997    0.110247
1998    0.085717
1999    0.073662
2000    0.072237
2001    0.078344
2002    0.087363
2003    0.092822
2004    0.098739
2005    0.107192
2006    0.126894
2007    0.161548
2008    0.170989
2009    0.181810
2010    0.200368
2011    0.230428
2012    0.214555
2013    0.208178
2014    0.222794
2015    0.239199
2016    0.260756
2017    0.272967
2018    0.294082
2019    0.309886
2020    0.319500
2021    0.334174
2022    0.351637
2023    0.37

## 5. Compustat 季度基本面

**构造目标：**
- `epspxq` → EVOL（盈余波动率）、EPERSIST（盈余持续性）
- `datadate` → LAG = ANNDATS − datadate
- `rdq` → ATT/NRANK 全市场公告日历（补充 I/B/E/S 覆盖不到的公司）


In [18]:
df = conn.raw_sql(f"""
    SELECT gvkey, datadate, fyearq, fqtr, rdq,
           epspxq, epspiq,
           ajexq, cshprq,
           saleq, ibq, atq, ltq, cshoq
    FROM comp.fundq
    WHERE indfmt = 'INDL'
      AND datafmt = 'STD'
      AND popsrc = 'D'
      AND consol = 'C'
      AND datadate BETWEEN '1992-01-01' AND '{END}'
""")
df.to_parquet(OUT / "compustat_quarterly.parquet", index=False)
print(f"compustat_quarterly: {len(df):,} rows | "
      f"{df['datadate'].min()} ~ {df['datadate'].max()}")


compustat_quarterly: 1,607,253 rows | 1992-01-31 ~ 2026-06-30


**`compustat_quarterly`（Compustat 季度基本面，来自 `comp.fundq`）**

| 字段 | English Full Name | 中文解释 | 对应 Design Doc 变量 |
|---|---|---|---|
| gvkey | Global Company Key | Compustat 公司标识符 | 公司标识（需转 PERMNO） |
| datadate | Data Date | 财季结束日期 | LAG = ANNDATS − datadate |
| fyearq | Fiscal Year (Quarterly) | 财政年度（季度口径） | — |
| fqtr | Fiscal Quarter | 财政季度序号 | — |
| rdq | Report Date of Quarterly Earnings | 季度盈余公告日期 | ATT/NRANK 全市场公告日历（补充 I/B/E/S 未覆盖公司） |
| epspxq | EPS Basic Excl. Extraordinary Items | 基本每股收益（不含非经常性项目） | EVOL、EPERSIST 构造核心字段 |
| epspiq | EPS Basic Incl. Extraordinary Items | 基本每股收益（含非经常性项目） | 备用 |
| ajexq | Adjustment Factor (Company) — Cumulative by Ex-Date, Quarterly | 季度累积拆股/股票股利调整因子 | EVOL / EPERSIST：`epspxq / ajexq` 折算到同一基准后才能做 16 季度季节差分 |
| cshprq | Common Shares Used to Calculate EPS — Basic, Quarterly | 计算基本 EPS 所用的普通股加权平均股数 | 备用：交叉验证 EPS 口径、或自行重算每股指标 |
| saleq | Sales/Turnover (Net) | 季度净销售额 | 备用控制指标 |
| ibq | Income Before Extraordinary Items | 非经常性项目前收益 | 备用 |
| atq | Total Assets (Quarterly) | 季度总资产 | 备用 |
| ltq | Total Liabilities (Quarterly) | 季度总负债 | 备用 |
| cshoq | Common Shares Outstanding (Quarterly) | 季度流通普通股数 | 备用 |


## 6. I/B/E/S Actuals（实际EPS + 公告日期）

**构造目标：** $e_{i,d}$（实际EPS）、ANNDATS（公告日，整个事件表的锚点）、ATT/NRANK 公告日历


In [19]:
cols = conn.describe_table(library='ibes', table='act_epsus')
print(cols)

Approximately 1323271 rows in ibes.act_epsus.
        name  nullable              type  \
0     ticker      True        VARCHAR(6)   
1      cusip      True        VARCHAR(8)   
2      oftic      True        VARCHAR(6)   
3      cname      True       VARCHAR(16)   
4      pends      True              DATE   
5    measure      True        VARCHAR(6)   
6    pdicity      True        VARCHAR(3)   
7    anndats      True              DATE   
8    anntims      True              TIME   
9    actdats      True              DATE   
10   acttims      True              TIME   
11     value      True  DOUBLE PRECISION   
12  curr_act      True        VARCHAR(3)   
13    usfirm      True          SMALLINT   

                                              comment  
0                                  IBES Ticker Symbol  
1                                         CUSIP/SEDOL  
2                              Official Ticker Symbol  
3                                        Company Name  
4            

In [20]:
df = conn.raw_sql(f"""
    SELECT ticker, cusip, cname, oftic,
           pends, anndats, anntims,
           value, curr_act
    FROM ibes.act_epsus
    WHERE pends BETWEEN '1995-07-01' AND '{END}'
      AND measure = 'EPS'
      AND curr_act = 'USD'
      AND pdicity = 'QTR'
      AND usfirm = 1
""")
df.to_parquet(OUT / "ibes_actuals.parquet", index=False)
print(f"ibes_actuals: {len(df):,} rows | "
      f"{df['anndats'].min()} ~ {df['anndats'].max()}")

ibes_actuals: 742,513 rows | 1995-04-01 ~ 2026-05-14


**`ibes_actuals`（I/B/E/S 实际EPS及公告日期，来自 `ibes.act_epsus`）**

| 字段 | English Full Name | 中文解释 | 对应 Design Doc 变量 |
|---|---|---|---|
| ticker | I/B/E/S Ticker Symbol | I/B/E/S 股票代码 | 需经 link_ibes_crsp 转 PERMNO |
| cusip | CUSIP/SEDOL | 证券识别码 | — |
| cname | Company Name | 公司名称 | — |
| oftic | Official Ticker Symbol | 官方股票代码 | — |
| pends | Period End Date | 财季结束日期 | — |
| anndats | Announce Date | 公告日期 | **事件表锚点 d**，NRANK/ATT 构造基础 |
| anntims | Announce Time | 公告时间 | 判断盘前/盘后 |
| value | Actual Value | 实际EPS数值 | e_{i,d}（SUE 分子） |
| curr_act | Currency (Company Level) | 币种 | 数据筛选（限定 USD） |


In [18]:
import pandas as pd
from pathlib import Path
OUT = Path("./data")

df3 = pd.read_parquet(OUT / "ibes_actuals.parquet")
df3.head()

,ticker,cusip,cname,oftic,pends,anndats,anntims,value,curr_act
0,ABCR,00075210,ABC RAIL PRODUCT,ABCR,1996-01-31,1996-02-27,00:00:00,0.33,USD
1,ABM,00095710,ABM INDUSTRIES,ABM,1996-01-31,1996-03-12,00:00:00,0.1,USD
2,ABRX,00077R10,ABR INFO SVCS,ABRX,1996-01-31,1996-02-21,00:00:00,0.075,USD
3,ABS,01310410,ALBERTSONS INC,ABS,1996-01-31,1996-03-04,00:00:00,0.61,USD
4,ABTE,00371210,ABLE TELECOM HLD,ABTE,1996-01-31,1996-03-04,00:00:00,-0.06,USD


## 7. I/B/E/S Detail（分析师逐条预测）

**构造目标：** $F_{i,d}$（共识预测，公告前60天窗口内分析师最新预测中位数）、LNANALYST

数据量较大，按年分批下载。


In [21]:
cols = conn.describe_table(library='ibes', table='det_epsus')
print(cols[['name','type','comment']])

Approximately 34540576 rows in ibes.det_epsus.
           name              type  \
0        ticker        VARCHAR(6)   
1         cusip        VARCHAR(8)   
2         oftic        VARCHAR(6)   
3         cname       VARCHAR(16)   
4       actdats              DATE   
5     estimator  DOUBLE PRECISION   
6        analys  DOUBLE PRECISION   
7        currfl        VARCHAR(1)   
8           pdf        VARCHAR(1)   
9           fpi        VARCHAR(1)   
10      measure        VARCHAR(3)   
11        value  DOUBLE PRECISION   
12         curr        VARCHAR(3)   
13       usfirm          SMALLINT   
14      fpedats              DATE   
15      acttims              TIME   
16      revdats              DATE   
17      revtims              TIME   
18      anndats              DATE   
19      anntims              TIME   
20       actual  DOUBLE PRECISION   
21  actdats_act              DATE   
22  acttims_act              TIME   
23  anndats_act              DATE   
24  anntims_act             

In [22]:
# 第一步：完全不筛选，看这张表本身能不能查到任意数据
test = conn.raw_sql("""
    SELECT ticker, fpedats, actdats, value, curr, fpi, measure, usfirm
    FROM ibes.det_epsus
    WHERE fpedats BETWEEN '2020-01-01' AND '2020-12-31'
    LIMIT 20
""")
print(test)

   ticker     fpedats     actdats  value  curr fpi measure  usfirm
0    00C6  2020-01-31  2019-11-26   1.08  <NA>   6     EPS       1
1    00C6  2020-01-31  2019-11-27   1.08  <NA>   6     EPS       1
2    00C6  2020-01-31  2019-11-27   1.07  <NA>   6     EPS       1
3    00C6  2020-01-31  2019-11-27   1.27  <NA>   6     EPS       1
4    00C6  2020-01-31  2019-11-27   1.08  <NA>   6     EPS       1
5    00C6  2020-01-31  2019-11-27   1.06  <NA>   6     EPS       1
6    00C6  2020-01-31  2019-11-28   1.08  <NA>   6     EPS       1
7    00C6  2020-01-31  2019-11-28   1.07  <NA>   6     EPS       1
8    00C6  2020-01-31  2019-08-21   1.02  <NA>   7     EPS       1
9    00C6  2020-01-31  2019-08-22   1.02  <NA>   7     EPS       1
10   00C6  2020-01-31  2019-08-22  1.167  <NA>   7     EPS       1
11   00C6  2020-01-31  2019-08-22    1.2  <NA>   7     EPS       1
12   00C6  2020-01-31  2019-08-22   1.03  <NA>   7     EPS       1
13   00C6  2020-01-31  2019-08-23   0.94  <NA>   7     EPS    

In [22]:
years = list(range(1995, 2027))

for yr in years:
    fpath = OUT / f"ibes_detail_{yr}.parquet"
    if fpath.exists():
        print(f"  已存在，跳过: ibes_detail_{yr}")
        continue
    try:
        df = conn.raw_sql(f"""
            SELECT ticker, cusip, cname,
                   analys, estimator,
                   fpedats, actdats, anndats, revdats,
                   value, curr, fpi,
                   measure, usfirm
            FROM ibes.det_epsus
            WHERE fpedats BETWEEN '{yr}-01-01' AND '{yr}-12-31'
              AND measure = 'EPS'
              AND fpi IN ('1','2','6','7')
              AND usfirm = 1
        """)
        df.to_parquet(fpath, index=False)
        print(f"  ibes_detail_{yr}: {len(df):,} rows")
    except Exception as e:
        print(f"  ✗ ibes_detail_{yr}: {e}")
    time.sleep(1)

print("I/B/E/S Detail 下载完成")

  ibes_detail_1995: 283,721 rows
  已存在，跳过: ibes_detail_1996
  已存在，跳过: ibes_detail_1997
  已存在，跳过: ibes_detail_1998
  已存在，跳过: ibes_detail_1999
  已存在，跳过: ibes_detail_2000
  已存在，跳过: ibes_detail_2001
  已存在，跳过: ibes_detail_2002
  已存在，跳过: ibes_detail_2003
  已存在，跳过: ibes_detail_2004
  已存在，跳过: ibes_detail_2005
  已存在，跳过: ibes_detail_2006
  已存在，跳过: ibes_detail_2007
  已存在，跳过: ibes_detail_2008
  已存在，跳过: ibes_detail_2009
  已存在，跳过: ibes_detail_2010
  已存在，跳过: ibes_detail_2011
  已存在，跳过: ibes_detail_2012
  已存在，跳过: ibes_detail_2013
  已存在，跳过: ibes_detail_2014
  已存在，跳过: ibes_detail_2015
  已存在，跳过: ibes_detail_2016
  已存在，跳过: ibes_detail_2017
  已存在，跳过: ibes_detail_2018
  已存在，跳过: ibes_detail_2019
  已存在，跳过: ibes_detail_2020
  已存在，跳过: ibes_detail_2021
  已存在，跳过: ibes_detail_2022
  已存在，跳过: ibes_detail_2023
  已存在，跳过: ibes_detail_2024
  已存在，跳过: ibes_detail_2025
  已存在，跳过: ibes_detail_2026
I/B/E/S Detail 下载完成


**`ibes_detail_{year}`（I/B/E/S 分析师逐条预测，来自 `ibes.det_epsus`）**

| 字段 | English Full Name | 中文解释 | 对应 Design Doc 变量 |
|---|---|---|---|
| ticker | I/B/E/S Ticker Symbol | I/B/E/S 股票代码 | 需经 link_ibes_crsp 转 PERMNO |
| cusip | CUSIP/SEDOL | 证券识别码 | — |
| cname | Company Name | 公司名称 | — |
| analys | Analyst Code | 分析师代码 | 同一分析师去重取最新一条；LNANALYST 计数 |
| estimator | Contributing Estimator Code | 提供预测的机构代码 | 备用 |
| fpedats | Forecast Period End Date | 预测目标财季结束日 | 判断是否为 1–2 季度前瞻预测 |
| actdats | Activation Date | 该条预测的发布/激活日期 | 60 天窗口对齐锚点（issued） |
| anndats | Announce Date | （若有）公告日期 | 备用对齐 |
| revdats | Review Date | 预测复核/更新日期 | 60 天窗口对齐（reviewed） |
| value | Estimate Value | 分析师预测的EPS数值 | F_{i,d} 的原始输入（取中位数前） |
| curr | Currency | 币种 | 本表该字段多为空值，未作筛选条件 |
| fpi | Forecast Period Indicator | 预测周期指标 | 筛选 1–2 季度前瞻预测（需进一步核实编码含义） |
| measure | Measure (Data Type Indicator) | 数据类型指标 | 筛选 = 'EPS' |
| usfirm | US Firm Flag | 是否美国公司数据源（1=是） | 样本筛选 |


In [25]:
df = pd.read_parquet(OUT / "ibes_detail_1996.parquet")

# 随机看几行
df.sample(10)

# 随机挑一个ticker，看它这一年所有的预测记录
random_ticker = df['ticker'].sample(1).values[0]
df[df['ticker'] == random_ticker].sort_values('actdats')

,ticker,cusip,cname,analys,estimator,fpedats,actdats,anndats,revdats,value,curr,fpi,measure,usfirm
217749,PRS2,<NA>,<NA>,20029.0,183.0,1996-12-31,1996-11-26,1996-11-26,1998-07-10,0.89,<NA>,1,EPS,1
217752,PRS2,<NA>,<NA>,20029.0,183.0,1996-12-31,1996-11-26,1996-11-26,1998-07-10,0.2,<NA>,6,EPS,1
217750,PRS2,<NA>,<NA>,8044.0,100.0,1996-12-31,1996-11-29,1996-11-29,1997-01-09,0.9,<NA>,1,EPS,1
217753,PRS2,<NA>,<NA>,8044.0,100.0,1996-12-31,1996-11-29,1996-11-29,1998-07-10,0.17,<NA>,6,EPS,1
217751,PRS2,74157E10,PRIME SERVICE,8044.0,100.0,1996-12-31,1997-01-10,1997-01-10,1998-07-10,0.85,<NA>,1,EPS,1


In [26]:
# 挑一条真实的公告记录做测试（用已经下载的ibes_actuals）
actuals = pd.read_parquet(OUT / "ibes_actuals.parquet")
sample_event = actuals[actuals['anndats'].between('1996-01-01','1996-06-30')].sample(1)
print(sample_event)

tkr = sample_event['ticker'].values[0]
anndate = pd.to_datetime(sample_event['anndats'].values[0])
window_start = anndate - pd.Timedelta(days=60)

# 找这只股票、在60天窗口内、针对这次财季的所有预测
preds = df[
    (df['ticker'] == tkr) &
    (pd.to_datetime(df['actdats']) >= window_start) &
    (pd.to_datetime(df['actdats']) < anndate)
]
print(f"窗口: {window_start.date()} ~ {anndate.date()}")
print(preds[['analys','actdats','value']].sort_values('actdats'))

     ticker     cusip         cname oftic       pends     anndats   anntims  \
2475   CVBK  15579210  CENT VA BCSH  CVBK  1996-03-31  1996-04-19  00:00:00   

      value curr_act  
2475  0.181      USD  
窗口: 1996-02-19 ~ 1996-04-19
       analys     actdats   value
75980  9613.0  1996-03-08  0.2221
75982  9613.0  1996-03-08  0.2262
75984  9613.0  1996-03-08  0.9132


## 7.5 I/B/E/S 拆股调整因子

**构造目标：** 把 actual EPS 与 60 天前的 forecast 折算到同一拆股基准。
I/B/E/S 的预测值和实际值各按记录当时的基准存储，跨越拆股的公告若直接相减，
(e − F) 会是两个不同基准的数。对应 Data Doc §3 表 6 末行 "earnings/forecasts/price 均需 split-adjusted"。


In [4]:
tabs = conn.list_tables(library='ibes')
print([t for t in tabs if 'adj' in t.lower()])


['adj', 'adjsum']


In [ ]:
# 7.5 I/B/E/S 拆股调整因子（配 detail 文件）
print(conn.describe_table(library='ibes', table='adj')[['name', 'type']])

df = conn.raw_sql("SELECT * FROM ibes.adj")
df.to_parquet(OUT / "ibes_adjustment.parquet", index=False)
print(f"\nibes_adjustment: {len(df):,} rows, cols={list(df.columns)}")
print(df.head())

Approximately 202959 rows in ibes.adj.
      name              type
0   ticker        VARCHAR(6)
1    cusip        VARCHAR(8)
2    oftic        VARCHAR(6)
3    cname       VARCHAR(16)
4  spdates              DATE
5      adj  DOUBLE PRECISION
6   usfirm          SMALLINT

ibes_adjustment: 202,959 rows, cols=['ticker', 'cusip', 'oftic', 'cname', 'spdates', 'adj', 'usfirm']
  ticker     cusip oftic             cname     spdates    adj  usfirm
0   0000  87482X10  TLMR    TALMER BANCORP  2014-02-20    1.0       1
1   0001  26878510   EPE         EP ENERGY  2014-02-20    1.0       1
2   0003  66515910   FFF   NORTHN FRONTIER  2014-02-20  1.019       0
3   0003  66515910   FFF   NORTHN FRONTIER  2014-07-17    1.0       0
4   0004  02504D10  ACSF  AMERICAN CAPITAL  2014-02-20    1.0       1


**`ibes_adjustment`（I/B/E/S 拆股调整因子，来自 `ibes.adj`）**

| 字段 | English Full Name | 中文解释 | 用途 |
|---|---|---|---|
| ticker | I/B/E/S Ticker Symbol | I/B/E/S 股票代码 | 与 `ibes_actuals` / `ibes_detail` 的连接键 |
| cusip | CUSIP | 证券识别码 | 备用连接键（→ PERMNO） |
| oftic | Official Ticker Symbol | 官方交易代码 | 人工核对用 |
| cname | Company Name | 公司名称 | 人工核对用 |
| spdates | Split Date (Effective Date of Factor) | 该调整因子的生效日期 | 判断某个每股数值处于哪个调整基准区间 |
| adj | Cumulative Adjustment Factor | 累积调整因子 | 把不同时点的每股数值折算到同一基准 |
| usfirm | US Firm Flag | 是否美国公司（1=是） | 样本筛选 |

**⚠️ 用法说明：本项目实际用不到这张表做 EPS 折算。**

实测确认 `ibes.act_epsus` 与 `ibes.det_epsus` **已经由 I/B/E/S 统一折算到最新的拆股基准**：
AAPL 2012Q2 的 actual EPS 在文件里是 `0.4393`，而当年实际披露的是 `$12.30`，
两者相差正好 28 倍 = 2014 年 7:1 与 2020 年 4:1 两次拆股的乘积。
既然 actual 和 forecast 出自同一份快照、同一基准，`(e − F)` 本身不存在基准错配问题。

这张表的价值在于两个场景：
1. 若后续改用**未调整版** `ibes.detu_epsus` / `ibes.actu_epsus`（可规避调整值的四舍五入损失），
   需要用它把每股数值折回公告当时的基准；
2. 交叉验证：确认某公司在样本期内的拆股事件与折算倍数。

**真正需要处理的是 SUE 的分母 P** —— 见下方说明。

## 8. Thomson Reuters 13F 机构持股

**构造目标：** IO = Σ 机构持股数 / 流通股本

按年分批下载。


In [5]:
cols = conn.describe_table(library='tfn', table='s34')
print(cols[['name','type','comment']])

Approximately 127142720 rows in tfn.s34.
        name              type                                      comment
0      fdate              DATE                                    File Date
1    mgrname       VARCHAR(45)                                 Manager Name
2    country       VARCHAR(44)                                      Country
3      mgrno  DOUBLE PRECISION                               Manager Number
4   typecode  DOUBLE PRECISION                                    Type Code
5      rdate              DATE                                  Report Date
6     prdate              DATE                            Prior Report Date
7      cusip        VARCHAR(8)                                        Cusip
8     shares  DOUBLE PRECISION                    Shares Held at End of Qtr
9       sole  DOUBLE PRECISION            Sole Voting Authority Shares Held
10    shared  DOUBLE PRECISION          Shared Voting Authority Shares Held
11        no  DOUBLE PRECISION              No 

In [6]:
years = list(range(1996, 2027))

for yr in years:
    fpath = OUT / f"tr13f_{yr}.parquet"
    if fpath.exists():
        print(f"  已存在，跳过: tr13f_{yr}")
        continue
    try:
        df = conn.raw_sql(f"""
            SELECT cusip, rdate, fdate, mgrno,
                   shares, prc AS holdings_prc,
                   shrout1, shrout2, typecode
            FROM tfn.s34
            WHERE fdate BETWEEN '{yr}-01-01' AND '{yr}-12-31'
        """)
        df.to_parquet(fpath, index=False)
        print(f"  tr13f_{yr}: {len(df):,} rows")
    except Exception as e:
        print(f"  ✗ tr13f_{yr}: {e}")
    time.sleep(1)

print("13F 下载完成")

  已存在，跳过: tr13f_1996
  已存在，跳过: tr13f_1997
  已存在，跳过: tr13f_1998
  已存在，跳过: tr13f_1999
  已存在，跳过: tr13f_2000
  已存在，跳过: tr13f_2001
  已存在，跳过: tr13f_2002
  已存在，跳过: tr13f_2003
  已存在，跳过: tr13f_2004
  已存在，跳过: tr13f_2005
  已存在，跳过: tr13f_2006
  已存在，跳过: tr13f_2007
  已存在，跳过: tr13f_2008
  已存在，跳过: tr13f_2009
  已存在，跳过: tr13f_2010
  已存在，跳过: tr13f_2011
  已存在，跳过: tr13f_2012
  已存在，跳过: tr13f_2013
  已存在，跳过: tr13f_2014
  已存在，跳过: tr13f_2015
  已存在，跳过: tr13f_2016
  已存在，跳过: tr13f_2017
  已存在，跳过: tr13f_2018
  已存在，跳过: tr13f_2019
  已存在，跳过: tr13f_2020
  已存在，跳过: tr13f_2021
  已存在，跳过: tr13f_2022
  已存在，跳过: tr13f_2023
  已存在，跳过: tr13f_2024
  已存在，跳过: tr13f_2025
  已存在，跳过: tr13f_2026
13F 下载完成


**`tr13f_{year}`（机构持股 13F，来自 `tfn.s34`）**

| 字段 | English Full Name | 中文解释 | 对应 Design Doc 变量 |
|---|---|---|---|
| cusip | CUSIP | 证券识别码 | 需转 PERMNO（经 crsp_security_info 的 cusip 字段） |
| rdate | Report Date | 报告期（该季度末持仓状态对应的日期） | — |
| fdate | File Date | 文件申报日期 | IO 取值时点 |
| mgrno | Manager Number | 机构管理人编号 | 用于对同一公司加总所有机构持股 |
| shares | Shares Held at End of Qtr | 该机构在季末持有的股数 | IO 分子（对 mgrno 求和） |
| holdings_prc（原字段名 prc） | Share Price, as of FDATE | 申报日股价 | 备用，交叉验证用 |
| shrout1 | Shares Outstanding (Millions) | 流通股本（百万股，13F自带口径） | IO 分母备选（建议优先用 CRSP shrout 保持口径一致） |
| shrout2 | Shares Outstanding (1000s) | 流通股本（千股，13F自带口径） | 同上 |
| typecode | Type Code | 持仓类型代码 | 备用 |


## 9. 下载结果汇总

In [23]:
print(f"{'文件名':<40s} {'大小(MB)':>10s}")
print("-" * 52)

total = 0
for f in sorted(OUT.glob("*.parquet")):
    mb = f.stat().st_size / 1024 / 1024
    total += mb
    print(f"{f.name:<40s} {mb:>9.1f}M")

print("-" * 52)
print(f"{'总计':<40s} {total:>9.1f}M")
print(f"\n存储位置: {OUT.resolve()}")


文件名                                          大小(MB)
----------------------------------------------------
compustat_annual.parquet                       9.4M
compustat_quarterly.parquet                   41.8M
crsp_daily_1996.parquet                       34.3M
crsp_daily_1997.parquet                       36.9M
crsp_daily_1998.parquet                       37.6M
crsp_daily_1999.parquet                       36.3M
crsp_daily_2000.parquet                       37.2M
crsp_daily_2001.parquet                       35.3M
crsp_daily_2002.parquet                       34.0M
crsp_daily_2003.parquet                       32.1M
crsp_daily_2004.parquet                       32.1M
crsp_daily_2005.parquet                       32.4M
crsp_daily_2006.parquet                       32.9M
crsp_daily_2007.parquet                       34.0M
crsp_daily_2008.parquet                       37.3M
crsp_daily_2009.parquet                       35.6M
crsp_daily_2010.parquet                       32.8M
crsp_daily_